In [1]:
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

BASE_DIR = Path("/Users/thunthita/Lidarforiypnb/LIDar/RawFile/")
OUT_DIR = Path("/Users/thunthita/Lidarforiypnb/LIDar/src/Pict/")
OUT_DIR.mkdir(exist_ok=True)

DAYS = [
    "05-02-2026",
    "06-02-2026",
    # "27-02-2026",
    # "19-02-2026",
    # "20-02-2026",
]

def process_minimpl(MiniMPL: pd.DataFrame) -> pd.DataFrame:
    nrb_max = MiniMPL["copol_nrb"].max()
    if not np.isfinite(nrb_max) or nrb_max == 0:
        norm = np.nan
    else:
        norm = MiniMPL["copol_nrb"] / nrb_max

    return pd.DataFrame({
        "range_raw": MiniMPL["range_raw"],
        "range_m_for_NRB": MiniMPL["range_raw"] * 1000,
        "range_m": MiniMPL["range_nrb"] * 1000,
        "copol_raw": MiniMPL["copol_raw"],
        "copol_snr": MiniMPL["copol_snr"],
        "copol_background": MiniMPL["copol_background"],
        "copol_nrb": MiniMPL["copol_nrb"],
        "crosspol_raw": MiniMPL["crosspol_raw"],
        "crosspol_snr": MiniMPL["crosspol_snr"],
        "crosspol_background": MiniMPL["crosspol_background"],
        "crosspol_nrb": MiniMPL["crosspol_nrb"],
        "laser_energy": MiniMPL["laser_energy"],
        "pbls": MiniMPL["pbls"],
        "Normalize_copol_nrb": norm,
    })

# =========================
# Main loop (per day)
# =========================
for day in DAYS:
    day_folder = BASE_DIR / f"{day}-DATfile"
    if not day_folder.exists():
        print(f"❌ Folder missing: {day_folder}")
        continue

    date_obj = pd.to_datetime(day, format="%d-%m-%Y")
    date_str = date_obj.strftime("%Y%m%d")

    times = pd.date_range(
        start=f"{date_str} 00:00",
        end=f"{date_str} 23:55",
        freq="5min"
    )

    print(f"\n📂 Processing {day}")

    rows_day = []          # ← reset every day
    missing_count = 0
    processed_count = 0

    for t in times:
        ts_str = t.strftime("%Y%m%d%H%M")
        infile = day_folder / f"MPL_5038_{ts_str}.csv"

        if not infile.exists():
            missing_count += 1
            print(f"❌ Missing file: {infile.name}")
            continue


        try:
            MiniMPL = pd.read_csv(infile)
        except Exception as e:
            print(f"⚠️ Read error: {infile.name} → {e}")
            continue

        df = process_minimpl(MiniMPL)
        df.insert(0, "timestamp", t)
        df.insert(0, "day", day)

        rows_day.append(df)
        processed_count += 1

    if rows_day:
        MiniMPL_day = pd.concat(rows_day, ignore_index=True)

        out_file = OUT_DIR / f"MiniMPL_{day}.csv"
        MiniMPL_day.to_csv(out_file, index=False)

        print(
            f"✅ {day}: saved {processed_count} files → {out_file.name} | "
            f"❌ missing {missing_count}"
        )
    else:
        print(f"⚠️ {day}: no valid data files")



📂 Processing 05-02-2026
✅ 05-02-2026: saved 288 files → MiniMPL_05-02-2026.csv | ❌ missing 0

📂 Processing 06-02-2026
✅ 06-02-2026: saved 288 files → MiniMPL_06-02-2026.csv | ❌ missing 0


In [2]:
# # Where your per-day combined CSVs are
# IN_DIR = Path("/Users/thunthita/Lidarforiypnb/LIDar/src/Pict/")

# # Where you want the extracted files to go
# OUT_DIR = Path("/Users/thunthita/Lidarforiypnb/LIDar/src/OutputPictCSV/CSVeachday/")
# OUT_DIR.mkdir(parents=True, exist_ok=True)


# for day in DAYS:
#     in_file = IN_DIR / f"MiniMPL_{day}.csv"
#     if not in_file.exists():
#         print(f"❌ Missing: {in_file}")
#         continue

#     df = pd.read_csv(in_file)

#     # Make sure timestamp is datetime
#     df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce")
#     df = df.dropna(subset=["timestamp"])

#     # Keep only times at minute 05 and 35 (00:05, 00:35, ... 23:35)
#     mask = df["timestamp"].dt.minute.isin([5, 35])

#     # Keep needed columns
#     df_sel = df.loc[mask, ["day", "timestamp", "pbls"]].copy()

#     # Reduce to 1 row per timestamp (since pbls repeats across range bins)
#     # Option A: take first value per timestamp
#     df_out = df_sel.groupby(["day", "timestamp"], as_index=False).first()

#     # Rename + create derived columns
#     df_out = df_out.rename(columns={"pbls": "pbls_km"})
#     df_out["pbls_m"] = df_out["pbls_km"] * 1000.0
#     df_out["min_pbls_m"] = df_out["pbls_m"] - 300.0
#     df_out["max_pbls_m"] = df_out["pbls_m"] + 300.0

#     # Reorder columns nicely
#     df_out = df_out[["day", "timestamp", "pbls_km", "pbls_m", "min_pbls_m", "max_pbls_m"]]

#     out_file = OUT_DIR / f"pbls_{day}_0005_0035_to_2335.csv"
#     df_out.to_csv(out_file, index=False)

#     print(f"✅ Saved: {out_file} | rows={len(df_out)}")


In [3]:
from pathlib import Path
import pandas as pd
import numpy as np

IN_DIR = Path("/Users/thunthita/Lidarforiypnb/LIDar/src/Pict/")
OUT_DIR = Path("/Users/thunthita/Lidarforiypnb/LIDar/src/OutputPictCSV/CSVeachday/")
OUT_DIR.mkdir(parents=True, exist_ok=True)

EXPECTED_TIMES = pd.to_timedelta(
    [f"{h:02d}:{m:02d}:00" for h in range(24) for m in (5, 35)]
)

for day in DAYS:
    in_file = IN_DIR / f"MiniMPL_{day}.csv"
    if not in_file.exists():
        print(f"❌ Missing: {in_file}")
        continue

    df = pd.read_csv(in_file)

    df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce")
    df = df.dropna(subset=["timestamp"])

    df["day"] = day

    mask = df["timestamp"].dt.minute.isin([5, 35])
    df_sel = df.loc[mask, ["day", "timestamp", "pbls"]].copy()

    df_sel = df_sel.groupby(["day", "timestamp"], as_index=False).first()

    # ✅ Correct parsing for DD-MM-YYYY
    day_ts = pd.to_datetime(day, format="%d-%m-%Y")

    full_index = pd.DataFrame({
        "day": day,
        "timestamp": day_ts + EXPECTED_TIMES
    })

    df_out = full_index.merge(df_sel, on=["day", "timestamp"], how="left")

    df_out = df_out.rename(columns={"pbls": "pbls_km"})
    df_out["pbls_m"] = df_out["pbls_km"] * 1000.0
    df_out["min_pbls_m"] = df_out["pbls_m"] - 300.0
    df_out["max_pbls_m"] = df_out["pbls_m"] + 300.0

    df_out = df_out[
        ["day", "timestamp", "pbls_km", "pbls_m", "min_pbls_m", "max_pbls_m"]
    ]

    out_file = OUT_DIR / f"pbls_{day}_0005_0035_to_2335.csv"
    df_out.to_csv(out_file, index=False)

    # Missing count (internal only)
    missing_count = df_out["pbls_km"].isna().sum()

    print(
        f"✅ Saved: {out_file.name} | rows={len(df_out)} "
        f"| missing={missing_count}"
    )


✅ Saved: pbls_05-02-2026_0005_0035_to_2335.csv | rows=48 | missing=0
✅ Saved: pbls_06-02-2026_0005_0035_to_2335.csv | rows=48 | missing=0


In [24]:
# #this code is only for the day that timestampo is 00.01 00.06 ... 23.56!!
# from pathlib import Path
# import pandas as pd
# import numpy as np

# IN_DIR = Path("/Users/thunthita/Lidarforiypnb/LIDar/src/Pict/")
# OUT_DIR = Path("/Users/thunthita/Lidarforiypnb/LIDar/src/OutputPictCSV/CSVeachday/")
# OUT_DIR.mkdir(parents=True, exist_ok=True)

# EXPECTED_TIMES = pd.to_timedelta(
#     [f"{h:02d}:{m:02d}:00" for h in range(24) for m in (6, 36)]
# )

# for day in DAYS:
#     in_file = IN_DIR / f"MiniMPL_{day}.csv"
#     if not in_file.exists():
#         print(f"❌ Missing: {in_file}")
#         continue

#     df = pd.read_csv(in_file)

#     df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce")
#     df = df.dropna(subset=["timestamp"])

#     df["day"] = day

#     # mask = df["timestamp"].dt.minute.isin([5, 35])
#     mask = df["timestamp"].dt.minute.isin([6, 36])
#     df_sel = df.loc[mask, ["day", "timestamp", "pbls"]].copy()

#     df_sel = df_sel.groupby(["day", "timestamp"], as_index=False).first()

#     # ✅ Correct parsing for DD-MM-YYYY
#     day_ts = pd.to_datetime(day, format="%d-%m-%Y")

#     full_index = pd.DataFrame({
#         "day": day,
#         "timestamp": day_ts + EXPECTED_TIMES
#     })

#     df_out = full_index.merge(df_sel, on=["day", "timestamp"], how="left")
#     df_out["timestamp"] = df_out["timestamp"] - pd.Timedelta(minutes=1)
#     df_out = df_out.rename(columns={"pbls": "pbls_km"})
#     df_out["pbls_m"] = df_out["pbls_km"] * 1000.0
#     df_out["min_pbls_m"] = df_out["pbls_m"] - 300.0
#     df_out["max_pbls_m"] = df_out["pbls_m"] + 300.0

#     df_out = df_out[
#         ["day", "timestamp", "pbls_km", "pbls_m", "min_pbls_m", "max_pbls_m"]
#     ]

#     out_file = OUT_DIR / f"pbls_{day}_sample_0006_0036_saved_as_0005_0035.csv"
#     df_out.to_csv(out_file, index=False)

#     # Missing count (internal only)
#     missing_count = df_out["pbls_km"].isna().sum()

#     print(
#         f"✅ Saved: {out_file.name} | rows={len(df_out)} "
#         f"| missing={missing_count}"
#     )


✅ Saved: pbls_23-02-2026_sample_0006_0036_saved_as_0005_0035.csv | rows=48 | missing=0
✅ Saved: pbls_26-02-2026_sample_0006_0036_saved_as_0005_0035.csv | rows=48 | missing=0
✅ Saved: pbls_27-02-2026_sample_0006_0036_saved_as_0005_0035.csv | rows=48 | missing=0


In [5]:
MINIMPL_DIR = Path("/Users/thunthita/Lidarforiypnb/LIDar/src/Pict")
PROTO_BASE = Path("/Users/thunthita/Lidarforiypnb/LIDar/src/OutputPictCSV")

for day in DAYS:
    minimipl_csv = MINIMPL_DIR / f"MiniMPL_{day}.csv"
    if not minimipl_csv.exists():
        print(f"❌ Missing MiniMPL CSV: {minimipl_csv.name}")
        continue

    # 👉 อ่าน MiniMPL ของวันนั้น
    MiniMPL_day = pd.read_csv(minimipl_csv, parse_dates=["timestamp"])
    MiniMPL_day["timestamp"] = MiniMPL_day["timestamp"].dt.floor("min")

    print(f"\n📅 Processing {day}")



📅 Processing 05-02-2026

📅 Processing 06-02-2026


In [7]:
MiniMPL_day.head()

,day,timestamp,range_raw,range_m_for_NRB,range_m,copol_raw,copol_snr,copol_background,copol_nrb,crosspol_raw,crosspol_snr,crosspol_background,crosspol_nrb,laser_energy,pbls,Normalize_copol_nrb
0,06-02-2026,2026-02-06,0.029979,29.979246,119.91698,23.479340,73486.125,0.0007,0.262720,18.154575,59153.9020,0.000665,0.018312,4.453778,1.948651,0.922162
1,06-02-2026,2026-02-06,0.059958,59.958490,149.89623,10.925777,34200.637,NaN,0.260519,1.406069,4578.8820,NaN,0.017618,NaN,NaN,0.914436
2,06-02-2026,2026-02-06,0.089938,89.937740,179.87548,10.674104,33416.910,NaN,0.262983,1.044370,3401.3381,NaN,0.018695,NaN,NaN,0.923085
3,06-02-2026,2026-02-06,0.119917,119.916980,209.85472,10.261624,32123.896,NaN,0.278234,0.887134,2888.6050,NaN,0.021449,NaN,NaN,0.976616
4,06-02-2026,2026-02-06,0.149896,149.896230,239.83397,9.758306,30544.139,NaN,0.284896,0.767723,2499.1692,NaN,0.022662,NaN,NaN,1.000000


In [9]:
def plot_profile_at_timestamp(
    df: pd.DataFrame,
    ts,
    *,
    xcol: str,
    ycol: str,
    figsize=(6, 3),
    xlabel=None,
    ylabel=None,
    title=None,
    xscale="linear",
    yscale="linear",
    xlim=None,
    ylim=None,
    show=False,
    save_path: Path | None = None,
    dpi=200,
    bbox_inches="tight",
):
    ts = pd.Timestamp(ts)
    df_t = df[df["timestamp"] == ts]

    if df_t.empty:
        return False  # reliable in loops

    if xcol not in df_t.columns or ycol not in df_t.columns:
        raise KeyError(f"Columns '{xcol}' or '{ycol}' not found")

    fig, ax = plt.subplots(figsize=figsize)
    ax.plot(df_t[xcol], df_t[ycol])

    ax.set_xlabel(xlabel if xlabel else xcol)
    ax.set_ylabel(ylabel if ylabel else ycol)

    if title:
        ax.set_title(title)
    else:
        ax.set_title(f"{ycol} vs {xcol}\n{ts}")

    ax.grid(True, alpha=0.3)
    ax.set_xscale(xscale)
    ax.set_yscale(yscale)

    if xlim:
        ax.set_xlim(*xlim)
    if ylim:
        ax.set_ylim(*ylim)

    fig.tight_layout()

    if save_path is not None:
        save_path.parent.mkdir(parents=True, exist_ok=True)
        fig.savefig(save_path, dpi=dpi, bbox_inches=bbox_inches)

    if show:
        plt.show()

    plt.close(fig)
    return True


In [11]:
OUT_DIR = Path("/Users/thunthita/Lidarforiypnb/LIDar/src/Pict")

PLOTS = [
    dict(ycol="copol_raw",           ylabel="Copol raw"),
    dict(ycol="copol_nrb",           ylabel="Copol nrb"),
    dict(ycol="Normalize_copol_nrb", ylabel="Normalize Copol nrb"),
]

for day in DAYS:
    daily_csv = OUT_DIR / f"MiniMPL_{day}.csv"
    if not daily_csv.exists():
        print(f"❌ Missing daily CSV: {daily_csv.name}")
        continue

    df_day = pd.read_csv(daily_csv, parse_dates=["timestamp"])

    # ✅ make timestamp matching robust (กันกรณีมีวินาที/เศษ)
    df_day["timestamp"] = df_day["timestamp"].dt.floor("min")

    # โฟลเดอร์ output ต่อวัน
    day_out = OUT_DIR / day
    day_out.mkdir(parents=True, exist_ok=True)

    # ✅ สร้าง date_str ต่อวัน (แก้จุดพังของคุณ)
    date_obj = pd.to_datetime(day, format="%d-%m-%Y")
    date_str = date_obj.strftime("%Y-%m-%d")

    # ทุก 30 นาที เริ่มที่ 00:05
    times_30min = pd.date_range(
        start=f"{date_str} 00:05",
        end=f"{date_str} 23:55",
        freq="30min"
    )

    saved = 0
    missing_ts = 0

    for ts in times_30min:
        ts = pd.Timestamp(ts).floor("min")  # ✅ ให้ match แน่
        hhmm = ts.strftime("%H%M")

        # (optional) ถ้า timestamp นั้นไม่มีจริง ก็ข้ามเร็ว ๆ
        if not (df_day["timestamp"] == ts).any():
            missing_ts += 1
            continue

        for p in PLOTS:
            ycol = p["ycol"]
            ylabel = p["ylabel"]

            out_png = day_out / f"{ycol}_{hhmm}.png"

            ok = plot_profile_at_timestamp(
                df_day,
                ts,
                xcol="range_m",
                ycol=ycol,
                xlabel="Range (m)",
                ylabel=ylabel,
                xlim=(0, 5000),
                show=False,
                save_path=out_png,
            )
            if ok:
                saved += 1

    print(f"✅ {day}: Saved {saved} plots → {day_out} | missing timestamps: {missing_ts}")


✅ 05-02-2026: Saved 144 plots → /Users/thunthita/Lidarforiypnb/LIDar/src/Pict/05-02-2026 | missing timestamps: 0
✅ 06-02-2026: Saved 144 plots → /Users/thunthita/Lidarforiypnb/LIDar/src/Pict/06-02-2026 | missing timestamps: 0


In [13]:
def plot_minimpl_vs_csv(
    MiniMPL_ALL: pd.DataFrame,
    *,
    ts,
    csv_path,
    xcol_minimpl,
    ycol_minimpl,
    xcol_csv,
    ycol_csv,
    label_minimpl="MiniMPL",
    label_csv="Prototype",
    figsize=(6, 3),
    xscale="linear",
    yscale="linear",
    xlim=None,
    ylim=None,
    title=None,
    show=True,
    save_path=None,          # <<< ADD
    find_peak=True,
    peak_xlim=(0, 5000),
    annotate_peak=False,
    peak_in_legend=False,
    peak_box=True,
    legend_loc="upper right",
):

    ts = pd.Timestamp(ts)
    csv_path = Path(csv_path)

    if not csv_path.exists():
        raise FileNotFoundError(csv_path)

    # ---- MiniMPL ----
    df_m = MiniMPL_ALL[MiniMPL_ALL["timestamp"] == ts]
    if df_m.empty:
        raise ValueError(f"No MiniMPL data for timestamp {ts}")

    # ---- External CSV ----
    df_c = pd.read_csv(csv_path)
    if xcol_csv not in df_c.columns or ycol_csv not in df_c.columns:
        raise KeyError("CSV columns not found")

    # ---- Convert to numpy + clean NaNs ----
    xm = df_m[xcol_minimpl].to_numpy()
    ym = df_m[ycol_minimpl].to_numpy()
    xc = df_c[xcol_csv].to_numpy()
    yc = df_c[ycol_csv].to_numpy()

    m_ok = np.isfinite(xm) & np.isfinite(ym)
    c_ok = np.isfinite(xc) & np.isfinite(yc)
    xm, ym = xm[m_ok], ym[m_ok]
    xc, yc = xc[c_ok], yc[c_ok]

    results = None

    # ---- Find peaks within peak_xlim ----
    if find_peak and peak_xlim is not None:
        lo, hi = peak_xlim
        m_mask = (xm >= lo) & (xm <= hi)
        c_mask = (xc >= lo) & (xc <= hi)

        if not np.any(m_mask):
            raise ValueError(f"No MiniMPL data in peak_xlim={peak_xlim}")
        if not np.any(c_mask):
            raise ValueError(f"No CSV data in peak_xlim={peak_xlim}")

        im = np.argmax(ym[m_mask])
        ic = np.argmax(yc[c_mask])

        peak_x_m = xm[m_mask][im]
        peak_y_m = ym[m_mask][im]
        peak_x_c = xc[c_mask][ic]
        peak_y_c = yc[c_mask][ic]

        results = {
            "ts": ts,
            "peak_xlim": peak_xlim,
            "minimpl_peak_x_m": float(peak_x_m),
            "minimpl_peak_y": float(peak_y_m),
            "csv_peak_x_m": float(peak_x_c),
            "csv_peak_y": float(peak_y_c),
            "peak_shift_m": float(peak_x_c - peak_x_m),  # csv - minimpl
        }

        print(f"[{ts}] Peak window: {lo}–{hi} m")
        print(f"  {label_minimpl} peak: x={peak_x_m:.1f} m, y={peak_y_m:.6g}")
        print(f"  {label_csv}     peak: x={peak_x_c:.1f} m, y={peak_y_c:.6g}")
        print(f"  Peak shift (csv - minimpl): {results['peak_shift_m']:.1f} m")

    # ---- Plot ----
    fig, ax = plt.subplots(figsize=figsize)

    # labels (optionally include peak info)
    lab_m = label_minimpl
    lab_c = label_csv
    if results and peak_in_legend:
        lab_m = f"{label_minimpl} (peak {results['minimpl_peak_x_m']:.0f} m)"
        lab_c = f"{label_csv} (peak {results['csv_peak_x_m']:.0f} m)"

    ax.plot(xm, ym, label=lab_m, linewidth=2)
    ax.plot(xc, yc, label=lab_c, linestyle="--")

    # draw peak markers
    if results:
        ax.axvline(results["minimpl_peak_x_m"], linestyle=":", linewidth=1.5)
        ax.axvline(results["csv_peak_x_m"], linestyle=":", linewidth=1.5)
        ax.scatter([results["minimpl_peak_x_m"]], [results["minimpl_peak_y"]], s=30)
        ax.scatter([results["csv_peak_x_m"]], [results["csv_peak_y"]], s=30)

        if annotate_peak:
            ax.text(
                results["minimpl_peak_x_m"], results["minimpl_peak_y"],
                f"{label_minimpl}\n{results['minimpl_peak_x_m']:.0f} m",
                fontsize=8, va="bottom", ha="left"
            )
            ax.text(
                results["csv_peak_x_m"], results["csv_peak_y"],
                f"{label_csv}\n{results['csv_peak_x_m']:.0f} m",
                fontsize=8, va="bottom", ha="right"
            )

        # peak summary box outside right (no fill)
        if peak_box:
            box_text = (
                f"Peak (within {results['peak_xlim'][0]}–{results['peak_xlim'][1]} m)\n"
                f"{label_minimpl}: {results['minimpl_peak_x_m']:.0f} m\n"
                f"{label_csv}: {results['csv_peak_x_m']:.0f} m\n"
                f"Shift (csv - minimpl): {results['peak_shift_m']:.0f} m"
            )
            ax.text(
                1.05, 0.98, box_text,
                transform=ax.transAxes,
                ha="left", va="top",
                fontsize=8,
                bbox=dict(
                    boxstyle="round,pad=0.3",
                    facecolor="none",
                    edgecolor="0.4",
                    linewidth=0.8
                )
            )

    ax.set_xlabel("Range [m]")
    ax.set_ylabel("Normalized Relative Backscatter (NRB)")
    ax.grid(True, alpha=0.3)
    ax.legend(loc=legend_loc)

    ax.set_xscale(xscale)
    ax.set_yscale(yscale)

    if xlim:
        ax.set_xlim(*xlim)
    if ylim:
        ax.set_ylim(*ylim)
    else:
        if yscale == "linear":
            ax.set_ylim(bottom=0)

    if title:
        ax.set_title(title)
    else:
        ax.set_title(f"{label_minimpl} vs {label_csv}\n{ts}")

    fig.tight_layout()

    if save_path is not None:
        save_path = Path(save_path)
        save_path.parent.mkdir(parents=True, exist_ok=True)
        fig.savefig(save_path, dpi=600, bbox_inches="tight")

    if show:
        plt.show()

    plt.close(fig)
    return results


In [15]:
MINIMPL_DIR = Path("/Users/thunthita/Lidarforiypnb/LIDar/src/Pict")
PROTO_BASE = Path("/Users/thunthita/Lidarforiypnb/LIDar/src/OutputPictCSV")

all_results = []

for day in DAYS:
    minimipl_csv = MINIMPL_DIR / f"MiniMPL_{day}.csv"
    if not minimipl_csv.exists():
        print(f"❌ Missing MiniMPL CSV: {minimipl_csv.name}")
        continue

    MiniMPL_day = pd.read_csv(minimipl_csv, parse_dates=["timestamp"])
    MiniMPL_day["timestamp"] = MiniMPL_day["timestamp"].dt.floor("min")

    # ✅ output folder ต่อวัน (ตรงนี้เท่านั้น!)
    out_dir = MINIMPL_DIR / day
    out_dir.mkdir(parents=True, exist_ok=True)

    # make the 30-min timestamps for this day
    date_obj = pd.to_datetime(day, format="%d-%m-%Y")
    date_str = date_obj.strftime("%Y-%m-%d")

    times_30min = pd.date_range(
        start=f"{date_str} 00:05",
        end=f"{date_str} 23:55",
        freq="30min"
    )

    print(f"\n📅 {day}: loop {len(times_30min)} timestamps")

    for ts in times_30min:
        ts = pd.Timestamp(ts).floor("min")
        hhmm = ts.strftime("%H%M")

        proto_csv = PROTO_BASE / f"{day}-{hhmm}" / f"{day}-{hhmm}_processed.csv"
        if not proto_csv.exists():
            print(f"⚠️ Missing Prototype CSV at {ts}")
            continue

        # ✅ output file ต่อ timestamp (ตรงนี้เท่านั้น!)
        out_png = out_dir / f"MiniMPL_vs_Prototype_{hhmm}.png"

        try:
            res = plot_minimpl_vs_csv(
                MiniMPL_day,
                ts=ts,
                csv_path=proto_csv,
                xcol_minimpl="range_m",
                ycol_minimpl="Normalize_copol_nrb",
                xcol_csv="range_m",
                ycol_csv="range2_norm",
                label_minimpl="MiniMPL",
                label_csv="Prototype",
                yscale="linear",
                xlim=(0, 15000),
                peak_xlim=(0, 15000),
                annotate_peak=False,
                peak_box=False,
                peak_in_legend=False,
                legend_loc="upper right",
                show=False,            # 👈 ไม่ popup
                save_path=out_png,     # 👈 save จริง
            )
            all_results.append(res)

        except ValueError as e:
            print(f"⚠️ Skip {ts}: {e}")



📅 05-02-2026: loop 48 timestamps
[2026-02-05 00:05:00] Peak window: 0–15000 m
  MiniMPL peak: x=209.9 m, y=1
  Prototype     peak: x=600.0 m, y=1
  Peak shift (csv - minimpl): 390.1 m
[2026-02-05 00:35:00] Peak window: 0–15000 m
  MiniMPL peak: x=329.8 m, y=1
  Prototype     peak: x=607.5 m, y=1
  Peak shift (csv - minimpl): 277.7 m
[2026-02-05 01:05:00] Peak window: 0–15000 m
  MiniMPL peak: x=119.9 m, y=1
  Prototype     peak: x=521.2 m, y=1
  Peak shift (csv - minimpl): 401.3 m
[2026-02-05 01:35:00] Peak window: 0–15000 m
  MiniMPL peak: x=329.8 m, y=1
  Prototype     peak: x=570.0 m, y=1
  Peak shift (csv - minimpl): 240.2 m
[2026-02-05 02:05:00] Peak window: 0–15000 m
  MiniMPL peak: x=119.9 m, y=1
  Prototype     peak: x=693.8 m, y=1
  Peak shift (csv - minimpl): 573.8 m
[2026-02-05 02:35:00] Peak window: 0–15000 m
  MiniMPL peak: x=119.9 m, y=1
  Prototype     peak: x=540.0 m, y=1
  Peak shift (csv - minimpl): 420.1 m
[2026-02-05 03:05:00] Peak window: 0–15000 m
  MiniMPL peak: